# HW1 Part IV Local Analysis

This notebook reads the completed Part IV run exported from Colab and creates report-ready tables and figures. It does not train models and does not recompute KID; it only analyzes the saved outputs under `part4_results_full_20260501_211900`.

In [ ]:
from __future__ import annotations

import math
from pathlib import Path

import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image, ImageDraw, ImageFont

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({
    "figure.dpi": 140,
    "savefig.dpi": 220,
    "font.size": 11,
    "axes.titlesize": 14,
    "axes.titleweight": "bold",
    "axes.labelsize": 11,
    "legend.fontsize": 10,
})

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "train.py").exists():
    REPO_ROOT = Path("/Users/eric/courses/cmu-10799-diffusion")

RESULT_DIR = REPO_ROOT / "HW1_CMU_10799_Spring_2026" / "part4_results_full_20260501_211900"
FIGURES_DIR = REPO_ROOT / "HW1_CMU_10799_Spring_2026" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

Q6_TABLE_PATH = RESULT_DIR / "evaluation" / "q6_parameterization" / "q6_kid_table.csv"
Q7_TABLE_PATH = RESULT_DIR / "evaluation" / "q7_sampling_steps" / "q7_kid_table.csv"

if not RESULT_DIR.exists():
    raise FileNotFoundError(f"Missing result directory: {RESULT_DIR}")
if not Q6_TABLE_PATH.exists():
    raise FileNotFoundError(f"Missing Q6 table: {Q6_TABLE_PATH}")
if not Q7_TABLE_PATH.exists():
    raise FileNotFoundError(f"Missing Q7 table: {Q7_TABLE_PATH}")

PREDICTION_ORDER = ["epsilon", "v", "x0", "score"]
COLORS = {
    "epsilon": "#4C72B0",
    "v": "#55A868",
    "x0": "#DD8452",
    "score": "#C44E52",
}

print("Result directory:", RESULT_DIR)
print("Figures directory:", FIGURES_DIR)

## Load KID Tables

KID is the main quantitative metric for Part IV. Lower KID is better. The standard deviation comes from the 10 KID subsets used during evaluation.

In [ ]:
q6_kid = pd.read_csv(Q6_TABLE_PATH)
q7_kid = pd.read_csv(Q7_TABLE_PATH)

metric_cols = [
    "kernel_inception_distance_mean",
    "kernel_inception_distance_std",
]
for df in (q6_kid, q7_kid):
    for col in metric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

q6_kid["prediction_type"] = pd.Categorical(
    q6_kid["prediction_type"], categories=PREDICTION_ORDER, ordered=True
)
q6_kid = q6_kid.sort_values("prediction_type").reset_index(drop=True)
q7_kid["num_steps"] = pd.to_numeric(q7_kid["num_steps"], errors="coerce").astype(int)
q7_kid = q7_kid.sort_values("num_steps").reset_index(drop=True)

q6_report_table = q6_kid[["prediction_type", *metric_cols]].copy()
q7_report_table = q7_kid[["num_steps", *metric_cols, "reused_from"]].copy()

q6_report_table.to_csv(FIGURES_DIR / "part4_q6_kid_table.csv", index=False)
q7_report_table.to_csv(FIGURES_DIR / "part4_q7_kid_table.csv", index=False)

print("Q6 KID table")
display(q6_report_table)
print("Q7 KID table")
display(q7_report_table)

## Q6: Parameterization KID Comparison

The score parameterization has a much larger KID, so the horizontal axis uses a log scale. This avoids visually hiding the differences among epsilon, v, and x0.

In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 4.2))
plot_df = q6_report_table.sort_values("kernel_inception_distance_mean", ascending=True)
y_pos = np.arange(len(plot_df))
bar_colors = [COLORS.get(str(name), "#4C72B0") for name in plot_df["prediction_type"].astype(str)]

ax.barh(
    y_pos,
    plot_df["kernel_inception_distance_mean"],
    xerr=plot_df["kernel_inception_distance_std"],
    color=bar_colors,
    alpha=0.9,
    capsize=4,
)
ax.set_yticks(y_pos)
ax.set_yticklabels(plot_df["prediction_type"].astype(str))
ax.set_xscale("log")
ax.set_xlabel("KID mean (log scale; lower is better)")
ax.set_title("Q6 parameterization comparison")
ax.grid(True, axis="x", alpha=0.3)
ax.grid(False, axis="y")
for i, (_, row) in enumerate(plot_df.iterrows()):
    mean = row["kernel_inception_distance_mean"]
    std = row["kernel_inception_distance_std"]
    ax.text(
        0.98,
        i,
        f"{mean:.4f} ± {std:.4f}",
        transform=ax.get_yaxis_transform(),
        ha="right",
        va="center",
        fontsize=9,
        bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.78, "pad": 1.5},
    )
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
fig.tight_layout()
output_path = FIGURES_DIR / "part4_q6_kid_bar.png"
fig.savefig(output_path, bbox_inches="tight")
plt.show()
print("Saved:", output_path)

## Q7: Sampling Steps Ablation

This plot compares deterministic strided sampling with fewer steps against the 1000-step baseline. Error bars are the KID subset standard deviation, so small ordering changes among 300-900 steps should not be over-interpreted.

In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 4.2))
ax.errorbar(
    q7_report_table["num_steps"],
    q7_report_table["kernel_inception_distance_mean"],
    yerr=q7_report_table["kernel_inception_distance_std"],
    marker="o",
    linewidth=2.0,
    capsize=4,
    color="#4C72B0",
)
ax.set_xlabel("Sampling steps")
ax.set_ylabel("KID mean (lower is better)")
ax.set_title("Q7 sampling steps ablation")
ax.set_xticks(q7_report_table["num_steps"].tolist())
ax.grid(True, alpha=0.3)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
for _, row in q7_report_table.iterrows():
    ax.annotate(
        f"{row['kernel_inception_distance_mean']:.4f}",
        (row["num_steps"], row["kernel_inception_distance_mean"]),
        textcoords="offset points",
        xytext=(0, 8),
        ha="center",
        fontsize=8,
    )
fig.tight_layout()
output_path = FIGURES_DIR / "part4_q7_kid_vs_steps.png"
fig.savefig(output_path, bbox_inches="tight")
plt.show()
print("Saved:", output_path)

## Training Curves

Raw loss is not directly comparable across parameterizations because each target has a different scale. The cross-parameterization training metric is `noise_mse`, computed after converting each model output back into a predicted noise tensor.

In [ ]:
def load_metrics(prediction_type: str) -> pd.DataFrame:
    metrics_path = RESULT_DIR / prediction_type / "metrics.csv"
    df = pd.read_csv(metrics_path)
    df["prediction_type"] = prediction_type
    return df

metrics_by_type = {prediction_type: load_metrics(prediction_type) for prediction_type in PREDICTION_ORDER}

fig, ax = plt.subplots(figsize=(8.2, 4.8))
for prediction_type, df in metrics_by_type.items():
    curve = df["noise_mse"].rolling(window=10, min_periods=1).mean()
    ax.plot(df["step"], curve, label=prediction_type, linewidth=1.8, color=COLORS[prediction_type])
ax.set_xlabel("Training step")
ax.set_ylabel("Noise MSE")
ax.set_title("Cross-parameterization noise MSE during training")
ax.legend(frameon=True)
ax.grid(True, alpha=0.3)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
fig.tight_layout()
output_path = FIGURES_DIR / "part4_noise_mse_curves.png"
fig.savefig(output_path, bbox_inches="tight")
plt.show()
print("Saved:", output_path)

final_rows = []
for prediction_type, df in metrics_by_type.items():
    last = df.iloc[-1]
    final_rows.append({
        "prediction_type": prediction_type,
        "step": int(last["step"]),
        "loss": float(last["loss"]),
        "noise_mse": float(last["noise_mse"]),
        "x0_mse": float(last["x0_mse"]),
        "noise_cosine": float(last["noise_cosine"]),
        "steps_per_sec": float(last["steps_per_sec"]),
        "gpu_memory_gb": float(last["gpu_memory_gb"]),
    })
final_metrics = pd.DataFrame(final_rows)
final_metrics.to_csv(FIGURES_DIR / "part4_final_training_metrics.csv", index=False)
display(final_metrics)

## Timestep-Bin Heatmap

Each cell is the average `noise_mse` over the final 10 logging points for one timestep interval. The color scale is logarithmic because low-noise and high-noise regions can differ substantially in magnitude.

In [ ]:
bin_columns = [col for col in metrics_by_type["epsilon"].columns if col.startswith("bin_") and col.endswith("_noise_mse")]
bin_labels = [col.replace("bin_", "").replace("_noise_mse", "").replace("_", "-") for col in bin_columns]
heatmap_rows = []
for prediction_type, df in metrics_by_type.items():
    late_average = df.tail(10)[bin_columns].mean(axis=0)
    heatmap_rows.append(late_average.to_numpy(dtype=float))
heatmap = np.vstack(heatmap_rows)

fig, ax = plt.subplots(figsize=(9.2, 3.8))
positive_values = heatmap[np.isfinite(heatmap) & (heatmap > 0)]
norm = mcolors.LogNorm(vmin=positive_values.min(), vmax=positive_values.max())
im = ax.imshow(heatmap, aspect="auto", cmap="viridis", norm=norm)
ax.set_yticks(np.arange(len(PREDICTION_ORDER)))
ax.set_yticklabels(PREDICTION_ORDER)
ax.set_xticks(np.arange(len(bin_labels)))
ax.set_xticklabels(bin_labels, rotation=35, ha="right")
ax.set_xlabel("Timestep bin")
ax.set_title("Late-training noise MSE by timestep bin")
cbar = fig.colorbar(im, ax=ax)
cbar.set_label("Noise MSE (log scale)")
for row_idx in range(heatmap.shape[0]):
    for col_idx in range(heatmap.shape[1]):
        value = heatmap[row_idx, col_idx]
        label = f"{value:.1e}" if value < 0.01 else f"{value:.3f}"
        ax.text(col_idx, row_idx, label, ha="center", va="center", fontsize=7, color="white")
fig.tight_layout()
output_path = FIGURES_DIR / "part4_timestep_bin_heatmap.png"
fig.savefig(output_path, bbox_inches="tight")
plt.show()
print("Saved:", output_path)

## Sample Grids

The next two cells assemble qualitative comparison panels from saved generated images. These are useful for the report because KID alone does not show the kind of visual artifact each setting produces.

In [ ]:
def add_label(image: Image.Image, label: str, pad: int = 30) -> Image.Image:
    image = image.convert("RGB")
    canvas = Image.new("RGB", (image.width, image.height + pad), "white")
    draw = ImageDraw.Draw(canvas)
    draw.text((8, 8), label, fill="black")
    canvas.paste(image, (0, pad))
    return canvas

def resize_width(image: Image.Image, width: int) -> Image.Image:
    height = max(1, round(image.height * width / image.width))
    return image.resize((width, height), Image.Resampling.LANCZOS)

def combine_grid(images: list[Image.Image], columns: int, background: str = "white") -> Image.Image:
    if not images:
        raise ValueError("No images to combine")
    tile_width = max(image.width for image in images)
    tile_height = max(image.height for image in images)
    rows = math.ceil(len(images) / columns)
    canvas = Image.new("RGB", (columns * tile_width, rows * tile_height), background)
    for index, image in enumerate(images):
        row = index // columns
        column = index % columns
        x = column * tile_width + (tile_width - image.width) // 2
        y = row * tile_height + (tile_height - image.height) // 2
        canvas.paste(image, (x, y))
    return canvas

q6_tiles = []
for prediction_type in PREDICTION_ORDER:
    sample_path = RESULT_DIR / prediction_type / "samples" / "samples_0100000.png"
    if not sample_path.exists():
        raise FileNotFoundError(sample_path)
    image = resize_width(Image.open(sample_path), 300)
    q6_tiles.append(add_label(image, prediction_type))

q6_panel = combine_grid(q6_tiles, columns=2)
q6_output = FIGURES_DIR / "part4_q6_final_samples.png"
q6_panel.save(q6_output)
display(q6_panel)
print("Saved:", q6_output)

In [ ]:
def localize_generated_dir(path_value: str) -> Path:
    # The CSV was produced in Colab, so it stores /content/drive paths.
    # For the extracted archive, keep only the suffix after full_20260501_211900.
    path_text = str(path_value)
    marker = "full_20260501_211900"
    if marker in path_text:
        suffix = path_text.split(marker, 1)[1].lstrip("/")
        return RESULT_DIR / suffix
    return Path(path_text)

def make_sample_strip(
    directory: Path,
    count: int = 8,
    image_size: int = 64,
    offset: int = 0,
) -> Image.Image:
    image_paths = sorted(directory.glob("*.png"))
    if len(image_paths) < offset + count:
        raise FileNotFoundError(
            f"Expected at least {offset + count} PNGs in {directory}, found {len(image_paths)}"
        )
    selected_paths = image_paths[offset : offset + count]
    images = [
        Image.open(path).convert("RGB").resize((image_size, image_size), Image.Resampling.LANCZOS)
        for path in selected_paths
    ]
    return combine_grid(images, columns=count)

q7_tiles = []
for row_index, (_, row) in enumerate(q7_report_table.iterrows()):
    generated_dir = localize_generated_dir(q7_kid.loc[q7_kid["num_steps"] == row["num_steps"], "generated_dir"].iloc[0])
    # Use different offsets per row so this panel is a representative sample
    # preview, not a same-index latent trajectory visualization.
    strip = make_sample_strip(generated_dir, count=8, image_size=64, offset=row_index * 8)
    q7_tiles.append(add_label(strip, f"{int(row['num_steps'])} steps"))

q7_panel = combine_grid(q7_tiles, columns=1)
q7_output = FIGURES_DIR / "part4_q7_step_samples.png"
q7_panel.save(q7_output)
display(q7_panel)
print("Saved:", q7_output)

## Short Report Takeaways

Use these points as the basis for the Part IV write-up. The wording can be tightened later inside `part_iv.tex`, but this cell keeps the quantitative claims tied to the generated tables.

In [ ]:
best_q6 = q6_report_table.sort_values("kernel_inception_distance_mean").iloc[0]
second_q6 = q6_report_table.sort_values("kernel_inception_distance_mean").iloc[1]
best_q7_less_than_1000 = q7_report_table[q7_report_table["num_steps"] < 1000].sort_values("kernel_inception_distance_mean").iloc[0]
full_1000 = q7_report_table[q7_report_table["num_steps"] == 1000].iloc[0]

summary_lines = [
    f"Q6 best parameterization: {best_q6['prediction_type']} with KID {best_q6['kernel_inception_distance_mean']:.6f} ± {best_q6['kernel_inception_distance_std']:.6f}.",
    f"Q6 second best: {second_q6['prediction_type']} with KID {second_q6['kernel_inception_distance_mean']:.6f} ± {second_q6['kernel_inception_distance_std']:.6f}.",
    f"Q7 best reduced-step sampler: {int(best_q7_less_than_1000['num_steps'])} steps with KID {best_q7_less_than_1000['kernel_inception_distance_mean']:.6f} ± {best_q7_less_than_1000['kernel_inception_distance_std']:.6f}.",
    f"Q7 1000-step baseline: KID {full_1000['kernel_inception_distance_mean']:.6f} ± {full_1000['kernel_inception_distance_std']:.6f}.",
    "Raw training loss should not be compared directly across parameterizations; noise_mse and KID are the safer cross-parameterization quantities.",
]
summary_path = FIGURES_DIR / "part4_summary_values.md"
summary_path.write_text("\n".join(f"- {line}" for line in summary_lines) + "\n")
for line in summary_lines:
    print("-", line)
print("Saved:", summary_path)